# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**What this notebook does (in simple words)**

We ask questions about one long PDF. There are two ways to find the right paragraphs ("chunks"):

* **Pipeline 1 (Baseline):** Turn every chunk into a vector (Bi-Encoder). Find the 10 closest chunks to the question. Give the **top 3** straight to the LLM.
* **Pipeline 2 (Re-Ranking):** Take the same 10 chunks, but let a **Cross-Encoder** read (question + chunk) together and re-score them. Give the **new top 3** to the LLM.

We then compare answers, rank changes, and speed.

**Stack:** Gemini API (embeddings + answers) | FAISS (vector store) | `cross-encoder/ms-marco-MiniLM-L-6-v2` (re-ranker)

**How to run:** Runtime > Run all. Before that, add your Gemini key in Colab: left sidebar > key icon (Secrets) > name `GEMINI_API_KEY`, and turn on "Notebook access". Get a free key at https://aistudio.google.com/apikey

## 1. Setup & Environment
Install the libraries we need (about 1 minute).

In [ ]:
# -q = quiet output. We install:
#  google-genai          -> talk to Gemini (embeddings + LLM answers)
#  pypdf                 -> read text out of the PDF
#  faiss-cpu             -> fast vector search (our vector store)
#  sentence-transformers -> gives us the Cross-Encoder model
#  pandas, numpy, requests -> tables, math, downloading
!pip install -q google-genai pypdf faiss-cpu sentence-transformers pandas numpy requests

In [ ]:
import os, re, time, json, getpass, textwrap
import numpy as np
import pandas as pd
import requests
import faiss
from pypdf import PdfReader
from google import genai
from google.genai import types
from sentence_transformers import CrossEncoder

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [ ]:
# ---------- SETTINGS (change these if you want) ----------
EMBED_BACKEND = "gemini"               # "gemini" (API, rate-limited on free tier) or "local" (free, runs inside Colab, no quota)
EMBED_MODEL = "gemini-embedding-001"   # Gemini embedding model (the Bi-Encoder)
EMBED_DIM   = 768                      # smaller vectors = faster + cheaper, still good quality
LOCAL_EMBED_MODEL = "BAAI/bge-small-en-v1.5"   # used only if EMBED_BACKEND = "local" (a real Bi-Encoder too)
# TIP: if you keep seeing "429 RESOURCE_EXHAUSTED", set EMBED_BACKEND = "local". The assignment only needs *a* dense Bi-Encoder.

# Free-tier friendly pacing for Gemini embeddings (free limit is roughly 100 embedded texts per minute)
EMBED_BATCH_SIZE   = 20     # texts sent per request
EMBED_PAUSE_SECS   = 18     # wait after each batch  (20 texts / 18s  ~= 65 texts per minute)
LLM_PAUSE_SECS     = 7      # wait after each answer (free Gemini Flash allows ~10 requests per minute)
LLM_MODEL   = "gemini-2.5-flash"       # Gemini model that writes the final answers
# If a model name gives a "not found" error, check https://ai.google.dev/gemini-api/docs/models and swap the name here.

CE_MODEL    = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # the Cross-Encoder re-ranker

CHUNK_WORDS   = 200   # how many words per chunk
CHUNK_OVERLAP = 40    # words shared between neighbouring chunks (so sentences are not cut badly)

TOP_K_RETRIEVE = 10   # candidates from the first stage (both pipelines)
TOP_K_CONTEXT  = 3    # chunks we actually give to the LLM

# ---------- API KEY ----------
# First try Colab Secrets. If not found, ask you to paste the key.
# OPTION A (most reliable): paste your key between the quotes below, run the cell.
# !! DELETE the key again (set it back to "") BEFORE you save/push this notebook to GitHub !!
MANUAL_KEY = ""

def _as_key_string(v):
    """Colab sometimes hands back the secret wrapped in a dict. This unwraps it into a plain string."""
    if isinstance(v, dict):
        print("  (Colab returned a dict with keys:", list(v.keys()), "- values are not printed)")
        for k in ("value", "key", "api_key", "payload", "GEMINI_API_KEY"):
            if isinstance(v.get(k), str):
                v = v[k]; break
        else:
            v = next((x for x in v.values() if isinstance(x, str)), None)
    return v.strip().strip('"').strip("'") if isinstance(v, str) and v.strip() else None

GEMINI_API_KEY = _as_key_string(MANUAL_KEY)
if not GEMINI_API_KEY:                      # OPTION B: Colab Secrets
    try:
        from google.colab import userdata
        GEMINI_API_KEY = _as_key_string(userdata.get("GEMINI_API_KEY"))
    except Exception as e:
        print("  Colab Secrets not readable:", type(e).__name__)
if not GEMINI_API_KEY:                      # OPTION C: environment variable
    GEMINI_API_KEY = _as_key_string(os.environ.get("GEMINI_API_KEY"))
if not GEMINI_API_KEY:                      # OPTION D: type it in
    print("Could not read the key automatically. Please paste it (it will not be saved in the notebook).")
    GEMINI_API_KEY = _as_key_string(getpass.getpass("Paste your Gemini API key: "))
if not GEMINI_API_KEY:
    raise RuntimeError("No usable API key. Paste it into MANUAL_KEY at the top of this cell (Option A), then run again.")

client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client ready.")

## 2. Dataset: one complex PDF
The document is the **EU GDPR** (Regulation 2016/679). **Upload it to Colab as `document.pdf` first** (left sidebar > Files). If that file is missing, the notebook tries to download it from EUR-Lex, and if that fails it asks you to upload it. A safety check stops the run if the PDF is not the GDPR.
GDPR is a great choice because it is full of rules, exceptions ("unless...", "except where..."), and prohibitions, which is exactly what we need for stress testing.

If every download fails (some sites block bots), the notebook asks you to upload any PDF yourself.

In [ ]:
# The document is pinned to GDPR so the notebook can never silently switch to another PDF.
# BEST OPTION: upload your GDPR PDF to Colab (left sidebar > Files > upload) and name it  document.pdf
# If document.pdf already exists, it is used and nothing is downloaded (EUR-Lex sometimes blocks downloads).
CANDIDATE_DOCS = [
    ("GDPR", "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679"),
    ("GDPR", "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L:2016:119:FULL"),
]
PDF_PATH = "document.pdf"
DOC_NAME = None
if os.path.exists(PDF_PATH):
    DOC_NAME = "GDPR"
    print("Found document.pdf in Colab, using it (no download needed).")

headers = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36"}

for name, url in ([] if DOC_NAME else CANDIDATE_DOCS):
    try:
        r = requests.get(url, headers=headers, timeout=90)
        # A real PDF always starts with the bytes "%PDF"
        if r.status_code == 200 and r.content[:4] == b"%PDF":
            with open(PDF_PATH, "wb") as f:
                f.write(r.content)
            DOC_NAME = name
            print(f"Downloaded {name} ({len(r.content)/1e6:.1f} MB) from {url}")
            break
        print(f"Skipped {url} (status {r.status_code}, not a PDF)")
    except Exception as e:
        print(f"Could not download {url}: {e}")

# Backup plan: upload your own PDF
if DOC_NAME is None:
    from google.colab import files
    print("Download failed. Please upload a PDF file now:")
    up = files.upload()
    uploaded_name = list(up.keys())[0]
    os.rename(uploaded_name, PDF_PATH)
    DOC_NAME = "GDPR"      # we expect the GDPR PDF; a safety check below confirms it
    print("Using your uploaded file:", uploaded_name)

In [ ]:
# Read the PDF page by page. We keep the page number so we can show it later.
reader = PdfReader(PDF_PATH)
pages = []
for page_no, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    text = re.sub(r"\s+", " ", text).strip()   # collapse messy whitespace/newlines
    if text:
        pages.append((page_no, text))

total_words = sum(len(t.split()) for _, t in pages)
print(f"Document: {DOC_NAME} | pages with text: {len(pages)} | total words: {total_words:,}")

# SAFETY CHECK: the GDPR is "Regulation (EU) 2016/679". If this number is missing, we loaded the wrong PDF.
full_text = " ".join(t for _, t in pages)
if not re.search(r"2016\s*/\s*679", full_text):
    raise RuntimeError("This PDF does not look like the GDPR. Upload the GDPR PDF as document.pdf "
                       "(left sidebar > Files), delete any old emb_*.npy / ckpt_*.npy files, and run again from the top.")
print("Check passed: this is the GDPR (Regulation (EU) 2016/679).")

## 3. Chunking
We cut the document into overlapping windows of words. **Both pipelines use these exact same chunks**, so the comparison is fair.

In [ ]:
def build_chunks(pages, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    """Slide a window of `size` words over the whole document, moving `size-overlap` words each step."""
    # Flatten document into a list of (word, page_number)
    words = [(w, pno) for pno, t in pages for w in t.split()]
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        window = words[start:start + size]
        # ignore a tiny leftover tail at the very end
        if len(window) < 30 and chunks:
            break
        chunks.append({
            "id": len(chunks),
            "page": window[0][1],                         # page where the chunk starts
            "text": " ".join(w for w, _ in window),
        })
    return chunks

chunks = build_chunks(pages)
chunk_texts = [c["text"] for c in chunks]
print(f"Created {len(chunks)} chunks. Example chunk #5 (page {chunks[5]['page']}):\n")
print(textwrap.fill(chunks[5]["text"][:500], 110), "...")

## 4. Stage 1 for both pipelines: Bi-Encoder embeddings + FAISS

A **Bi-Encoder** turns each chunk into a vector on its own (no question involved). We do this once, store the vectors, and later compare the question vector to them with cosine similarity.

* Gemini wants a *task type*: `RETRIEVAL_DOCUMENT` for chunks and `RETRIEVAL_QUERY` for questions.
* We normalise vectors to length 1, so **inner product = cosine similarity**. FAISS `IndexFlatIP` does exact inner-product search.

In [ ]:
def with_retry(fn, tries=8):
    """Free API keys have rate limits (error 429). If we hit one, wait the time Google asks for, then try again."""
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            msg = str(e)
            # Google often says "Please retry in 23.4s", so we read that number and wait that long
            m = re.search(r"retry in ([\d.]+)s", msg) or re.search(r"retryDelay'?\"?:\s*'?\"?(\d+)", msg)
            wait = float(m.group(1)) + 2 if m else min(15 * (i + 1), 65)
            if "PerDay" in msg or "per day" in msg.lower():
                raise RuntimeError("Daily free quota is used up. Set EMBED_BACKEND = 'local' (free) or try tomorrow / another key.") from e
            print(f"  API error ({msg[:70]}...). Waiting {wait:.0f}s then retrying ({i+1}/{tries})")
            time.sleep(wait)
    raise RuntimeError("API kept failing. Check your key / quota / model name, or set EMBED_BACKEND = 'local'.")

# ---- local embedding model (only loaded if you choose the local backend) ----
_local_model = None
def get_local_model():
    global _local_model
    if _local_model is None:
        from sentence_transformers import SentenceTransformer
        _local_model = SentenceTransformer(LOCAL_EMBED_MODEL)
    return _local_model

BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # bge models like this prefix on queries

def embed_texts(texts, task_type, checkpoint=None):
    """Turn texts into normalised vectors.
    task_type: "RETRIEVAL_DOCUMENT" (chunks) or "RETRIEVAL_QUERY" (questions).
    checkpoint: optional file name; progress is saved there, so if the run stops you can just re-run and it resumes."""
    if EMBED_BACKEND == "local":
        model = get_local_model()
        if task_type == "RETRIEVAL_QUERY":
            texts = [BGE_QUERY_PREFIX + t for t in texts]
        mat = model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=len(texts) > 50)
        return np.array(mat, dtype="float32")

    # ---- Gemini backend (slow & careful so we do not hit the free-tier limit) ----
    all_vecs = []
    if checkpoint and os.path.exists(checkpoint):
        all_vecs = list(np.load(checkpoint))
        print(f"  Resuming: {len(all_vecs)}/{len(texts)} already embedded.")
    for i in range(len(all_vecs), len(texts), EMBED_BATCH_SIZE):
        batch = texts[i:i + EMBED_BATCH_SIZE]
        resp = with_retry(lambda: client.models.embed_content(
            model=EMBED_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=EMBED_DIM),
        ))
        all_vecs.extend([e.values for e in resp.embeddings])
        if checkpoint:
            np.save(checkpoint, np.array(all_vecs, dtype="float32"))   # save progress after every batch
        if len(texts) > EMBED_BATCH_SIZE:                              # no need to wait for a single query
            print(f"  embedded {len(all_vecs)}/{len(texts)}", end="\r")
            time.sleep(EMBED_PAUSE_SECS)
    mat = np.array(all_vecs, dtype="float32")
    faiss.normalize_L2(mat)       # make each vector length 1 (so inner product = cosine similarity)
    return mat

# Cache so you do not wait again if you re-run the cell
tag = f"{EMBED_BACKEND}_{DOC_NAME}_{len(chunks)}"
cache_file = f"emb_{tag}.npy"
ckpt_file = f"ckpt_{tag}.npy"
if os.path.exists(cache_file):
    chunk_vecs = np.load(cache_file)
    print("Loaded embeddings from cache.")
else:
    t0 = time.perf_counter()
    est = len(chunks) / EMBED_BATCH_SIZE * EMBED_PAUSE_SECS / 60 if EMBED_BACKEND == "gemini" else 0
    if est: print(f"Embedding {len(chunks)} chunks with Gemini. Paced for the free tier: about {est:.0f} minutes. Please wait...")
    chunk_vecs = embed_texts(chunk_texts, "RETRIEVAL_DOCUMENT", checkpoint=ckpt_file)
    np.save(cache_file, chunk_vecs)
    print(f"\nEmbedded {len(chunks)} chunks in {time.perf_counter()-t0:.1f}s")

# Build the vector store (dimension is read from the vectors, so it works for both backends)
EMBED_DIM = chunk_vecs.shape[1]
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(chunk_vecs)
print("FAISS index size:", index.ntotal, "| vector dimension:", chunk_vecs.shape[1])

## 5. The two pipelines
Below are small, well-labelled functions so each step is easy to follow.

In [ ]:
# Load the Cross-Encoder once (downloads ~90 MB the first time)
cross_encoder = CrossEncoder(CE_MODEL)
print("Cross-Encoder loaded:", CE_MODEL)

# ---------- STAGE 1 (shared by both pipelines) ----------
def retrieve_top_k(query, k=TOP_K_RETRIEVE):
    """Embed the query, search FAISS. Returns chunk ids, cosine scores, and timings."""
    t0 = time.perf_counter()
    q_vec = embed_texts([query], "RETRIEVAL_QUERY")       # step a: embed the question (API call)
    t1 = time.perf_counter()
    scores, ids = index.search(q_vec, k)                  # step b: top-k by cosine similarity
    t2 = time.perf_counter()
    return {
        "ids": ids[0].tolist(),
        "scores": scores[0].tolist(),
        "t_embed": t1 - t0,
        "t_search": t2 - t1,
        "t_total": t2 - t0,
    }

# ---------- STAGE 2 (only Pipeline 2) ----------
def rerank(query, candidate_ids):
    """Score each (query, chunk) pair with the Cross-Encoder, sort high -> low."""
    t0 = time.perf_counter()
    pairs = [(query, chunks[i]["text"]) for i in candidate_ids]
    ce_scores = cross_encoder.predict(pairs)              # joint reading of query + chunk
    order = np.argsort(-ce_scores)                        # best first
    t1 = time.perf_counter()
    return {
        "ids": [candidate_ids[j] for j in order],
        "scores": [float(ce_scores[j]) for j in order],
        "t_rerank": t1 - t0,
    }

# ---------- LLM (identical prompt for both pipelines) ----------
PROMPT_TEMPLATE = """You are a careful assistant answering questions about a document.
Use ONLY the context chunks below. If the answer is not in the context, say "The provided context does not contain the answer."
Mention any exceptions or conditions that appear in the context. Keep the answer short (max 5 sentences).

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

def build_prompt(query, chunk_ids):
    context = "\n\n".join(f"[Chunk {i} | page {chunks[i]['page']}]\n{chunks[i]['text']}" for i in chunk_ids)
    return PROMPT_TEMPLATE.format(context=context, question=query)

def generate_answer(query, chunk_ids):
    prompt = build_prompt(query, chunk_ids)
    timing = {}
    def _call():
        t0 = time.perf_counter()                 # restarted on every attempt, so retry waits (e.g. 503 errors) are NOT counted
        r = client.models.generate_content(
            model=LLM_MODEL, contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0),   # 0 = same answer each time, fair comparison
        )
        timing["dt"] = time.perf_counter() - t0  # time of the successful call only
        return r
    resp = with_retry(_call)
    time.sleep(LLM_PAUSE_SECS)                   # stay under the free-tier requests-per-minute limit
    return (resp.text or "").strip(), timing["dt"]

# Warm-up so the first timed run is not unfairly slow (model loading etc.)
_ = rerank("warm up", list(range(10)))
print("Functions ready.")

## 6. Quick demo on one question
Shows both pipelines side by side so you can see the idea before the stress tests.

In [ ]:
def run_both(query, with_llm=True):
    """Run Pipeline 1 and Pipeline 2 for one query and return everything in one dict."""
    base = retrieve_top_k(query)                         # Top 10 (shared)
    rr = rerank(query, base["ids"])                      # Cross-Encoder over those 10

    base_top3 = base["ids"][:TOP_K_CONTEXT]              # Pipeline 1 context
    rr_top3 = rr["ids"][:TOP_K_CONTEXT]                  # Pipeline 2 context

    out = {"query": query, "base": base, "rr": rr, "base_top3": base_top3, "rr_top3": rr_top3}
    if with_llm:
        out["ans1"], out["t_llm1"] = generate_answer(query, base_top3)
        out["ans2"], out["t_llm2"] = generate_answer(query, rr_top3)
    return out

def rank_shift_table(res):
    """Table: for all 10 candidates, baseline rank vs. cross-encoder rank."""
    base_rank = {cid: r + 1 for r, cid in enumerate(res["base"]["ids"])}
    rows = []
    for new_rank, (cid, sc) in enumerate(zip(res["rr"]["ids"], res["rr"]["scores"]), start=1):
        b = base_rank[cid]
        rows.append({
            "chunk_id": cid, "page": chunks[cid]["page"],
            "baseline_rank": b, "reranked_rank": new_rank,
            "move": b - new_rank,                      # positive = moved UP
            "cosine": round(res["base"]["scores"][b - 1], 3),
            "cross_enc_score": round(sc, 2),
        })
    return pd.DataFrame(rows)

demo = run_both("How long does a company have to report a personal data breach to the authority?"
                if DOC_NAME == "GDPR" else "What are the main functions of the AI RMF Core?")
print("QUERY:", demo["query"], "\n")
display(rank_shift_table(demo))
print("\n--- ANSWER 1 (baseline top-3) ---\n", demo["ans1"])
print("\n--- ANSWER 2 (re-ranked top-3) ---\n", demo["ans2"])

## 7. Failure Mode Analysis (stress testing)

Here we run many tricky queries **without the LLM first** (cheap and fast) and let the notebook find where re-ranking really changes things.

A query is flagged as a **failure candidate** when a chunk that was at baseline rank **4 to 10** gets promoted into the cross-encoder's **Top 3**. That means the baseline would have hidden it from the LLM.

Query categories (from the assignment):
* **Keyword Trap**: lots of shared words with a wrong passage, but the real answer uses different wording
* **Exception**: a general rule exists, but a special "except when..." is elsewhere
* **Negation**: "not allowed", "prohibited", limits
* **Buried**: any other case where a key chunk sits low in the list

You can edit or add your own queries in `CUSTOM_QUERIES` (format: `("Category", "your question")`).

In [ ]:
GDPR_QUERIES = [
    # --- Negation / prohibition ---
    ("Negation",     "Is it forbidden to process data revealing someone's religious beliefs or political opinions?"),
    ("Negation",     "Who may not be subject to a decision based solely on automated processing?"),
    ("Negation",     "In which cases is a data subject not entitled to have their data erased?"),
    # --- Exceptions & corner cases ---
    ("Exception",    "Under what conditions can the ban on processing special categories of data be lifted?"),
    ("Exception",    "Are small organisations with fewer than 250 employees exempt from keeping records of processing?"),
    ("Exception",    "Can personal data be sent to a country without an adequacy decision?"),
    ("Exception",    "From what age can a child consent to online services without a parent?"),
    # --- Keyword trap (query words match a wrong passage, answer uses other terms) ---
    ("Keyword trap", "What is the highest penalty a company can be charged for breaking the core processing principles?"),
    ("Keyword trap", "How fast must a company tell the regulator when personal information is leaked?"),
    ("Keyword trap", "When is a company required to appoint someone to oversee data protection compliance?"),
    # --- Other / buried ---
    ("Buried",       "Can a person demand to receive their data in a format they can move to another service?"),
    ("Buried",       "What are the rules for handling data about criminal convictions?"),
]

NIST_QUERIES = [
    ("Negation",     "What do trustworthy AI systems not guarantee, according to the framework?"),
    ("Exception",    "Does the framework apply to all organisations equally, or are there limits to its use?"),
    ("Keyword trap", "How should organisations measure the risks of AI systems over time?"),
    ("Keyword trap", "What does the Govern function cover?"),
    ("Buried",       "Which characteristics make an AI system valid and reliable?"),
    ("Buried",       "How are AI actors defined in the document?"),
    ("Exception",    "What challenges make AI risk measurement difficult?"),
    ("Negation",     "What should a organisation not assume about risk tolerance?"),
]

# Add your own here, they are always included
CUSTOM_QUERIES = [
    # ("Exception", "Your own question here"),
]

if DOC_NAME == "GDPR":
    QUERIES = GDPR_QUERIES + CUSTOM_QUERIES
elif DOC_NAME == "NIST_AI_RMF":
    QUERIES = NIST_QUERIES + CUSTOM_QUERIES
else:
    QUERIES = CUSTOM_QUERIES
    assert QUERIES, "You uploaded your own PDF, so please fill CUSTOM_QUERIES above with 8-12 tricky questions."

print(f"{len(QUERIES)} queries ready.")

In [ ]:
# ---------- AUTO-SCAN (retrieval only, no LLM calls yet) ----------
scan_rows, scan_results = [], {}

for cat, q in QUERIES:
    base = retrieve_top_k(q)
    rr = rerank(q, base["ids"])
    base_rank = {cid: r + 1 for r, cid in enumerate(base["ids"])}

    rr_top3 = rr["ids"][:TOP_K_CONTEXT]
    # chunks that the cross-encoder put in top 3, but baseline had at rank 4..10
    promoted = [(cid, base_rank[cid]) for cid in rr_top3 if base_rank[cid] > TOP_K_CONTEXT]
    # chunks that baseline had in top 3, but cross-encoder dropped
    dropped = [cid for cid in base["ids"][:TOP_K_CONTEXT] if cid not in rr_top3]
    best_jump = max([(r - (rr_top3.index(cid) + 1)) for cid, r in promoted], default=0)

    scan_results[q] = {"category": cat, "base": base, "rr": rr}
    scan_rows.append({
        "category": cat,
        "query": q[:85] + ("..." if len(q) > 85 else ""),
        "full_query": q,
        "baseline_top3": base["ids"][:3],
        "reranked_top3": rr_top3,
        "promoted (chunk: old rank)": ", ".join(f"#{c}: {r}->in top3" for c, r in promoted) or "-",
        "n_promoted": len(promoted),
        "best_jump": best_jump,
        "failure_candidate": len(promoted) > 0,
        "t_baseline_s": base["t_total"],
        "t_rerank_s": rr["t_rerank"],
    })

scan_df = pd.DataFrame(scan_rows)
display(scan_df[["category", "query", "baseline_top3", "reranked_top3",
                 "promoted (chunk: old rank)", "failure_candidate"]])
print(f"\n{int(scan_df.failure_candidate.sum())} of {len(scan_df)} queries are failure candidates "
      f"(a chunk from rank 4-10 got promoted to the top 3).")

## 8. Deep dive on the best failure cases
We now pick the failure candidates with the **biggest rank jump** (default: top 4, covering different categories when possible) and run the full pipeline *with* the LLM.

You can also set `MANUAL_PICK` to a list of your own query strings to inspect specific ones.

In [ ]:
N_DEEP_DIVE = 4    # only used when MANUAL_PICK is empty (auto mode)
# Pre-filled: the queries with the biggest rank jumps, spread over the failure types.
# Set MANUAL_PICK = [] to let the notebook auto-pick instead. Strings must match the query list exactly.
MANUAL_PICK = [
    "Is it forbidden to process data revealing someone's religious beliefs or political opinions?",   # Negation, chunk #10 -> top 3
    "Who may not be subject to a decision based solely on automated processing?",                   # Negation, chunks #6 and #10 -> top 3
    "In which cases is a data subject not entitled to have their data erased?",                     # Negation, chunk #8 -> top 3
    "Can personal data be sent to a country without an adequacy decision?",                         # Exception
    "Can a person demand to receive their data in a format they can move to another service?",      # Buried
    "What is the highest penalty a company can be charged for breaking the core processing principles?",  # Keyword trap
]

if MANUAL_PICK:
    picked = MANUAL_PICK
else:
    cands = scan_df[scan_df.failure_candidate].sort_values("best_jump", ascending=False)
    picked, seen_cat = [], set()
    # first pass: one query per category (variety), second pass: fill the rest
    for _, row in cands.iterrows():
        if row.category not in seen_cat and len(picked) < N_DEEP_DIVE:
            picked.append(row.full_query); seen_cat.add(row.category)
    for _, row in cands.iterrows():
        if row.full_query not in picked and len(picked) < N_DEEP_DIVE:
            picked.append(row.full_query)

print("Selected queries for deep dive:")
for p in picked: print(" -", p)

In [ ]:
deep_results = {}
for q in picked:
    res = run_both(q, with_llm=True)
    res["category"] = scan_results[q]["category"] if q in scan_results else "Manual"
    deep_results[q] = res

    print("=" * 110)
    print(f"[{res['category']}] QUERY: {q}")
    print("=" * 110)
    display(rank_shift_table(res))

    # Plain-English rank shift sentences for the report
    base_rank = {cid: r + 1 for r, cid in enumerate(res["base"]["ids"])}
    for new_rank, cid in enumerate(res["rr_top3"], start=1):
        old = base_rank[cid]
        if old != new_rank:
            print(f"  -> Chunk {cid} (page {chunks[cid]['page']}) moved from baseline rank #{old} to re-ranked #{new_rank}")

    print("\nTop-3 chunk previews AFTER re-ranking:")
    for cid in res["rr_top3"]:
        print(f"  [Chunk {cid}, page {chunks[cid]['page']}] " + textwrap.shorten(chunks[cid]["text"], 260))

    print("\n--- ANSWER 1 (baseline top-3) ---\n" + textwrap.fill(res["ans1"], 110))
    print("\n--- ANSWER 2 (re-ranked top-3) ---\n" + textwrap.fill(res["ans2"], 110))
    print("\n")
    time.sleep(1)

### Your observations (write these in your own words for the report)
For each deep-dive query above, read **Answer 1** and **Answer 2** and note:
1. Which chunk moved the most (old rank -> new rank)?
2. Did Answer 1 miss a clause, state a wrong rule, or say "context does not contain the answer"? Did Answer 2 fix it?
3. Which failure type was it (keyword trap, exception, negation, buried)?

Sometimes re-ranking does **not** help or even hurts. If you see that, mention it honestly. It shows real understanding.

## 9. Latency profiling
We time each stage. Numbers come from the scan above (all queries) so they are averages, not a single lucky run.

* **Baseline retrieval** = embed the query (API call) + FAISS search
* **Re-ranking** = Cross-Encoder scoring 10 pairs
* **LLM** = answer generation (measured in the deep dive)

In [ ]:
# Detailed baseline timings
embed_t  = [v["base"]["t_embed"]  for v in scan_results.values()]
search_t = [v["base"]["t_search"] for v in scan_results.values()]
base_t   = [v["base"]["t_total"]  for v in scan_results.values()]
rerank_t = [v["rr"]["t_rerank"]   for v in scan_results.values()]
llm_t    = [r["t_llm1"] for r in deep_results.values()] + [r["t_llm2"] for r in deep_results.values()]

def ms(x): return np.array(x) * 1000

latency = pd.DataFrame([
    {"Stage": "Pipeline 1: query embedding (API)",       "Mean (ms)": ms(embed_t).mean(),  "Median (ms)": np.median(ms(embed_t)),  "Max (ms)": ms(embed_t).max()},
    {"Stage": "Pipeline 1: FAISS search (top 10)",       "Mean (ms)": ms(search_t).mean(), "Median (ms)": np.median(ms(search_t)), "Max (ms)": ms(search_t).max()},
    {"Stage": "Pipeline 1 total retrieval",              "Mean (ms)": ms(base_t).mean(),   "Median (ms)": np.median(ms(base_t)),   "Max (ms)": ms(base_t).max()},
    {"Stage": "Pipeline 2 extra: Cross-Encoder rerank",  "Mean (ms)": ms(rerank_t).mean(), "Median (ms)": np.median(ms(rerank_t)), "Max (ms)": ms(rerank_t).max()},
    {"Stage": "Pipeline 2 total retrieval (P1 + rerank)","Mean (ms)": (ms(base_t)+ms(rerank_t)).mean(), "Median (ms)": np.median(ms(base_t)+ms(rerank_t)), "Max (ms)": (ms(base_t)+ms(rerank_t)).max()},
    {"Stage": "LLM answer generation (either pipeline)", "Mean (ms)": ms(llm_t).mean(),    "Median (ms)": np.median(ms(llm_t)),    "Max (ms)": ms(llm_t).max()},
]).round(1)
display(latency)

overhead = ms(rerank_t).mean()
print(f"\nRe-ranking adds about {overhead:.0f} ms per query on top of baseline retrieval "
      f"({overhead / ms(base_t).mean() * 100:.0f}% of baseline retrieval time).")
print("Note: runtime depends on hardware. Re-run on CPU vs GPU in Colab to see the difference "
      "(Runtime > Change runtime type).")

In [ ]:
# Optional: how does Cross-Encoder time grow with the number of candidates?
# This helps answer the "scaling question" with real numbers.
test_q = QUERIES[0][1]
rows = []
for n in [10, 50, 100, min(500, len(chunks))]:
    ids = list(range(min(n, len(chunks))))
    t0 = time.perf_counter()
    cross_encoder.predict([(test_q, chunks[i]["text"]) for i in ids])
    rows.append({"chunks scored by Cross-Encoder": len(ids), "time (ms)": round((time.perf_counter() - t0) * 1000, 1)})
scale_df = pd.DataFrame(rows)
scale_df["ms per chunk"] = (scale_df["time (ms)"] / scale_df["chunks scored by Cross-Encoder"]).round(2)
display(scale_df)

per_chunk = scale_df["ms per chunk"].iloc[-1]
for N in [10_000, 1_000_000]:
    print(f"Estimated time to score {N:,} chunks for ONE query at {per_chunk} ms/chunk: ~{N*per_chunk/1000/60:,.1f} minutes")

## 10. Report helper (fill this in with YOUR OWN words)
The assignment says the report must be in your own words, so this notebook does **not** write it for you. It gives you the facts and a skeleton.

**1. Architecture comparison**: hints to think about
* Bi-Encoder: query and chunk go through the model *separately*, vectors are compared with cosine. Chunk vectors can be computed once and stored.
* Cross-Encoder: query and chunk are fed *together* as one input, so attention can connect words in the query to words in the chunk (this is how it can notice "not", "except", etc.).

**2. Rank shift analysis**: copy numbers from Section 8 (for each failure query: chunk X moved from rank #a to #b) and compare Answer 1 vs Answer 2.

**3. Latency table**: copy the table from Section 9.

**4. Scaling question**: hints to think about
* How many chunks would a real collection have? Use the "estimated time" printed above.
* What can be precomputed with a Bi-Encoder but not with a Cross-Encoder?
* What does "two-stage retrieval" trade off (recall vs precision vs speed)?

In [ ]:
# Save results to CSV files (handy to attach to your GitHub repo / report)
scan_df.drop(columns=["full_query"]).to_csv("failure_scan_results.csv", index=False)
latency.to_csv("latency_table.csv", index=False)
print("Saved failure_scan_results.csv and latency_table.csv (see the Files panel on the left in Colab).")